In [1]:
fileName = 'data/the-verdict.txt'

rawText = None
with open(fileName, 'r', encoding='utf-8') as f:
    rawText = f.read()
    
print('Total number of character: ', len(rawText))
print(rawText[:99])

Total number of character:  20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [2]:
#
# 1. Tokenization.
#

import re

preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', rawText)
preprocessed = [item for item in preprocessed if item.strip() != '']
print(len(preprocessed))
print(preprocessed[:30])

4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


In [3]:
#
# 2. Creating a vocabulary.
#

allWords = sorted(set(preprocessed))
vocabSize = len(allWords)
print('Vocabulary size: ', vocabSize)

vocab = {word: i for i, word in enumerate(allWords)}
for i, item in enumerate(vocab.items()):
    print(item)
    if i > 50:
        break

Vocabulary size:  1130
('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)
('His', 51)


In [4]:
#
# 3. Converting tokens to IDs.
#

from typing import List, Dict

class SimpleTokenizerV1:
    
    def encode(self, text: str) -> List[int]:
        tokens = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        tokens = [item for item in tokens if item.strip() != '']
        return [self.__vocab[token] for token in tokens]

    def decode(self, ids: List[int]) -> str:
        return ' '.join([self.__reverseVocab[i] for i in ids])
    
    def __init__(self, vocab: Dict[str, int]) -> None:
        self.__vocab = vocab
        self.__reverseVocab = {i: word for word, i in vocab.items()}
    
tokenizer = SimpleTokenizerV1(vocab)
text = """
"It's the last he painted, you know,"
Mrs. Gisburn said with pardonable pride.
"""

ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]
" It ' s the last he painted , you know , " Mrs . Gisburn said with pardonable pride .


In [5]:
#
# 2.4. Adding special tokens - Handling unknown words.
#

allTokens = sorted(set(preprocessed))
allTokens.extend(['<|endoftext|>', '<|unk|>'])
vocab = {token: integer for integer, token in enumerate(allTokens)}

print('Vocabulary size: ', len(vocab))

for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)
    if i > 50:
        break


Vocabulary size:  1132
('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [6]:
import re

from typing import List, Dict

class SimpleTokenizerV2(object):

    def encode(self, text: str) -> List[int]:
        preProcessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preProcessed = [item.strip() for item in preProcessed if item.strip() != '']
        preProcessed = [item if item in self.__strToInt else '<|unk|>' for item in preProcessed]
        ids = [self.__strToInt[item] for item in preProcessed]
        return ids

    def decode(self, ids: List[int]) -> str:
        text = ' '.join([self.__intToStr[id] for id in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r'\1', text)
        return text

    def __init__(self, vocab: Dict[str, int]) -> None:
        self.__strToInt = vocab
        self.__intToStr = {i: s for s, i in vocab.items()}
        
        
text1 = 'Hello, do you like tea?'
text2 = 'In the sunlit terraces of the palace.'
text = ' <|endoftext|> '.join((text1, text2))
print(text)

tokenizer = SimpleTokenizerV2(vocab)

ids = tokenizer.encode(text)
print(ids)
s = tokenizer.decode(ids)
print(s)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.


In [7]:
#
# 2.5. Byte Pair Encoding (BPE)
#

from importlib.metadata import version

import tiktoken
print('tiktoken version: ', version('tiktoken'))

tokenizer = tiktoken.get_encoding('gpt2')

text = 'Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunkownPalace.'
ids = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
print(ids)

s = tokenizer.decode(ids)
print(s)


tiktoken version:  0.14.0
[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 2954, 593, 11531, 558, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terraces of someunkownPalace.


In [8]:
#
# 2.6. Data sampling with a sliding window.
#

#
# Generating the input-target pairs to train the language model.
#

fileName = 'data/the-verdict.txt'
with open(fileName, 'r', encoding='utf-8') as f:
    rawText = f.read()
    
ids = tokenizer.encode(rawText, allowed_special={'<|endoftext|>'})
print('Total number of tokens: ', len(ids))

idsSampled = ids[50:]

contextSize = 4
x = idsSampled[:contextSize]
y = idsSampled[1:contextSize+1]
print(f'x: {x}')
print(f'y:      {y}')
print()

for i in range(1, contextSize + 1):
    context = idsSampled[:i]
    target = idsSampled[i]
    print(f'Context: {context} --> Target: {target}')
    
print()

for i in range(1, contextSize + 1):
    contextS = tokenizer.decode(idsSampled[:i])
    targetS = tokenizer.decode([idsSampled[i]])
    print(f'Context: {contextS} --> Target: {targetS}')


Total number of tokens:  5145
x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]

Context: [290] --> Target: 4920
Context: [290, 4920] --> Target: 2241
Context: [290, 4920, 2241] --> Target: 287
Context: [290, 4920, 2241, 287] --> Target: 257

Context:  and --> Target:  established
Context:  and established --> Target:  himself
Context:  and established himself --> Target:  in
Context:  and established himself in --> Target:  a


In [10]:
#
# A dataset for batched inputs and targets.
#

from typing import Tuple

import torch
from torch.utils.data import Dataset

class GPTDatasetV1(Dataset):
    
    def __init__(
        self, 
        text: str, 
        tokenizer: tiktoken.core.Encoding, 
        maxLength: int, 
        stride: int
    ) -> None:
        self.__inputIds = []
        self.__targetIds = []
        
        tokenIds = tokenizer.encode(text)
        for i in range(0, len(tokenIds) - maxLength, stride):
            inputChunk = tokenIds[i:i + maxLength]
            targetChunk = tokenIds[i + 1:i + maxLength + 1]
            
            self.__inputIds.append(torch.tensor(inputChunk))
            self.__targetIds.append(torch.tensor(targetChunk))
            
    def __len__(self) -> int:
        return len(self.__inputIds)
    
    def __getitem__(self, index: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.__inputIds[index], self.__targetIds[index]

In [11]:
from torch.utils.data import DataLoader

def createDataLoaderV1(
    text: str,
    batchSize: int=4,
    maxLength: int=256,
    stride: int=128,
    shuffle: bool=True,
    dropLast: bool=True,
    numWorkers: int=0
):
    tokenizer = tiktoken.get_encoding('gpt2')
    dataSet = GPTDatasetV1(text, tokenizer, maxLength, stride)
    dataLoader = DataLoader(
        dataSet,
        batch_size=batchSize,
        shuffle=shuffle,
        drop_last=dropLast,
        num_workers=numWorkers
    )
    
    return dataLoader

In [13]:
fileName = 'data/the-verdict.txt'

rawText = None
with open(fileName, 'r', encoding='utf-8') as f:
    rawText = f.read()
    
dataLoader = createDataLoaderV1(
    rawText,
    batchSize=1,
    maxLength=4,    # Context size.
    stride=1,
    shuffle=False
)

dataIter = iter(dataLoader)
firstBatch = next(dataIter)
print('First batch:', firstBatch)

secondBatch = next(dataIter)
print('Second batch:', secondBatch)

First batch: [tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]
Second batch: [tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [14]:
dataLoader = createDataLoaderV1(
    rawText,
    batchSize=8,
    maxLength=4,    # Context size.
    stride=4,
    shuffle=False
)

dataIter = iter(dataLoader)
inputs, targets = next(dataIter)

print('Inputs:', inputs)
print('Targets:', targets)

Inputs: tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets: tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])
